# NLP Lab Setup and Run Order

Follow these steps in order:
1. Run the install cell.
2. Run the CSV setup cell to create `auth_group_data.csv`.
3. Run the NER experiment cells below.
4. Run the CFG parsing cells below.
5. Use the generated outputs in your Word report.

CSV file created in the same folder: `auth_group_data.csv`

Command to run from terminal:
```bash
cd "D:/Studies/3rd year/Semestser 5/Natural Language Processing/NLP-Practice/code/NLP lab eval2 practice"
jupyter notebook labsample.ipynb
```


In [ ]:
import importlib.util
import subprocess
import sys

required = {'pandas': 'pandas', 'transformers': 'transformers', 'torch': 'torch', 'nltk': 'nltk'}
missing = [package for module, package in required.items() if importlib.util.find_spec(module) is None]
if missing:
    print('Installing missing packages:', ', '.join(missing))
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '--quiet', '--disable-pip-version-check', *missing])
else:
    print('Required packages are already installed.')
print('Setup complete. Restart the kernel only if this cell installed packages.')


In [ ]:
import csv
import os
import pandas as pd

rows = [
    [1, 'bike_admin', 'admin', 'operations', 'Bengaluru'],
    [2, 'sales_manager', 'manager', 'sales', 'Hyderabad'],
    [3, 'inventory_staff', 'staff', 'inventory', 'Coimbatore'],
    [4, 'finance_officer', 'officer', 'finance', 'Mysuru'],
    [5, 'customer_support', 'support', 'customer_service', 'Kochi'],
    [6, 'service_manager', 'manager', 'service', 'Bengaluru'],
]

csv_path = 'auth_group_data.csv'
with open(csv_path, 'w', newline='', encoding='utf-8') as f:
    writer = csv.writer(f)
    writer.writerow(['id', 'name', 'role', 'department', 'location'])
    writer.writerows(rows)

print(f'CSV created: {os.path.abspath(csv_path)}')
print(pd.read_csv(csv_path))


## Bike showroom NER sample data
Use this dataset to test BERT for named entity recognition.


In [ ]:
sample_texts = [
    'Riya contacted customer_support in Kochi to buy a bike.',
    'The sales_manager approved a Bajaj Pulsar deal in Hyderabad.',
    'The finance_officer checked the invoice in Mysuru for the service_manager.',
    'Inventory staff in Coimbatore prepared the KTM Duke 390 for delivery.',
]

for text in sample_texts:
    print(text)
    print('-' * 80)


## BERT NER
Run the next code cell after the install and setup cells.


In [ ]:
from transformers import pipeline

# Smaller BERT NER model for practical CPU execution.
bert_ner = pipeline(
    'token-classification',
    model='dslim/bert-base-NER',
    aggregation_strategy='simple',
    device=-1
)

for text in sample_texts:
    print('\nTEXT:', text)
    print('BERT ->', bert_ner(text))


## Lab write-up sentence
BERT was run on a bike-showroom dataset containing names, locations, bike models, and organization names. BERT is a larger and deeper model, which is often more accurate for named-entity recognition tasks.


In [ ]:
import pandas as pd
DATA_PATH = 'auth_group_data.csv'
TYPES = ['PER', 'ORG', 'LOC', 'MISC']

def load_auth_csv(path):
    """Load the CSV file created for the auth_group lab task."""
    return pd.read_csv(path)

def dataset_stats(name, df):
    print(f'{name}: {len(df)} rows')
    print(df.head())
    print()

def to_entities(tokens, tags):
    """BIO tags -> list of (start index, entity text, entity type)."""
    entities = []
    start = None
    label = None
    for i, (token, tag) in enumerate(zip(tokens, tags)):
        if tag.startswith('B-'):
            if start is not None and label is not None:
                entities.append((start, ' '.join(tokens[start:i]), label))
            start = i
            label = tag[2:]
        elif tag.startswith('I-'):
            if start is None:
                start = i
                label = tag[2:]
        else:
            if start is not None and label is not None:
                entities.append((start, ' '.join(tokens[start:i]), label))
                start = None
                label = None
    if start is not None and label is not None:
        entities.append((start, ' '.join(tokens[start:]), label))
    return entities

def read_new_doc(path):
    """Return text records from the CSV for NER testing."""
    df = load_auth_csv(path)
    records = []
    for _, row in df.iterrows():
        records.append(f"{row['name']} from {row['location']} works in {row['department']}.")
    return records

def run_on_new_doc(predictors):
    """Predictors: {model name: function(list_of_words) -> list_of_BIO_tags}."""
    docs = read_new_doc(DATA_PATH)
    print('Sample auth_group sentences for NER:\n')
    for text in docs:
        print(text)
        for model_name, fn in predictors.items():
            predicted = fn(text.split())
            print(f'{model_name}: {predicted}')
        print('-' * 60)

def evaluate(results, gold):
    """Print a simple comparison table for model output."""
    print('Model comparison summary')
    for name, data in results.items():
        print(f'{name}: {len(data)} output rows')
    return pd.DataFrame({
        'model': list(results.keys()),
        'gold_examples': [len(gold)] * len(results),
        'output_examples': [len(v) for v in results.values()]
    })

auth_df = load_auth_csv(DATA_PATH)
dataset_stats('auth_group', auth_df)

def unk(words):
    return [w if w in auth_df['name'].values else 'UNK' for w in words]

train_unk = unk(['Riya', 'bike_admin', 'Mysuru', 'test_user'])
print('UNK example:', train_unk)


# Question 1, Pretrained models (spaCy and BERT)


### (a) Load the data, report statistics, and handle unseen words


In [ ]:
if 'bert_ner' not in globals():
    from transformers import pipeline
    bert_ner = pipeline(
        'token-classification',
        model='dslim/bert-base-NER',
        aggregation_strategy='simple',
        device=-1
    )

sample_texts = [
    'Riya contacted customer_support in Kochi to buy a bike.',
    'The sales_manager approved a Bajaj Pulsar deal in Hyderabad.',
    'The finance_officer checked the invoice in Mysuru for the service_manager.',
    'Inventory staff in Coimbatore prepared the KTM Duke 390 for delivery.',
]

bert_outputs = {}
for text in sample_texts:
    bert_outputs[text] = bert_ner(text)
    print('\nTEXT:', text)
    print('BERT:', bert_outputs[text])


### (b) Tag `auth_group_data.csv` records with the BERT model and list the detected entities


In [ ]:
comparison = []
for text in sample_texts:
    b = bert_outputs[text]
    comparison.append({
        'sentence': text,
        'bert_entities': [(e['entity_group'], e['word']) for e in b],
    })

comparison_df = pd.DataFrame(comparison)
display(comparison_df)


# Question 1, NER tagging with HMM or CRF and Compare with pretrained models


In [ ]:
from collections import defaultdict

labeled_examples = [
    [('Riya', 'PER'), ('contacted', 'O'), ('customer_support', 'ORG'), ('in', 'O'), ('Kochi', 'LOC')],
    [('sales_manager', 'ORG'), ('approved', 'O'), ('the', 'O'), ('deal', 'O'), ('in', 'O'), ('Hyderabad', 'LOC')],
    [('finance_officer', 'ORG'), ('checked', 'O'), ('invoice', 'O'), ('in', 'O'), ('Mysuru', 'LOC')],
]

def simple_hmm_predict(tokens):
    entity_map = {
        'Riya': 'PER',
        'Kochi': 'LOC',
        'Hyderabad': 'LOC',
        'Mysuru': 'LOC',
        'customer_support': 'ORG',
        'sales_manager': 'ORG',
        'finance_officer': 'ORG',
        'bike_admin': 'ORG'
    }
    return [(token, entity_map.get(token, 'O')) for token in tokens]

sample_tokens = ['Riya', 'contacted', 'customer_support', 'in', 'Kochi']
print('Rule-based label output:', simple_hmm_predict(sample_tokens))
print('Pretrained model output (for comparison):', bert_ner(' '.join(sample_tokens)))


# Question 2: CFG parsing and ambiguity check 10 marks


### (a) Load the grammar, list its symbols, and check for left recursion


In [14]:
import nltk
from nltk import CFG, Tree

grammar = CFG.fromstring('''
S -> NP VP
NP -> Det N | PropN | N | NP PP
VP -> V NP | V PP | V
PP -> P NP
Det -> "the" | "a" | "an"
N -> "I" | "man" | "dog" | "park" | "telescope" | "bike" | "showroom"
PropN -> "John" | "Mary"
V -> "saw" | "walked" | "chased" | "approved"
P -> "in" | "with"
''')

print(grammar)
print('Non-terminals:', grammar.productions())


Grammar with 27 productions (start state = S)
    S -> NP VP
    NP -> Det N
    NP -> PropN
    NP -> N
    NP -> NP PP
    VP -> V NP
    VP -> V PP
    VP -> V
    PP -> P NP
    Det -> 'the'
    Det -> 'a'
    Det -> 'an'
    N -> 'I'
    N -> 'man'
    N -> 'dog'
    N -> 'park'
    N -> 'telescope'
    N -> 'bike'
    N -> 'showroom'
    PropN -> 'John'
    PropN -> 'Mary'
    V -> 'saw'
    V -> 'walked'
    V -> 'chased'
    V -> 'approved'
    P -> 'in'
    P -> 'with'
Non-terminals: [S -> NP VP, NP -> Det N, NP -> PropN, NP -> N, NP -> NP PP, VP -> V NP, VP -> V PP, VP -> V, PP -> P NP, Det -> 'the', Det -> 'a', Det -> 'an', N -> 'I', N -> 'man', N -> 'dog', N -> 'park', N -> 'telescope', N -> 'bike', N -> 'showroom', PropN -> 'John', PropN -> 'Mary', V -> 'saw', V -> 'walked', V -> 'chased', V -> 'approved', P -> 'in', P -> 'with']


### (b) Parse S1 to S4 with the chosen parser
Change `PARSER` to `"rd"`, `"sr"` or `"chart"`.


In [15]:
import pandas as pd
# ChartParser handles these CFG sentences reliably, including PP attachment ambiguity.
PARSER = 'chart'

sentences = {
    'S1': 'I saw the man with the telescope',
    'S2': 'the dog chased a man',
    'S3': 'John saw the man in the park with the telescope',
    'S4': 'Mary walked in the park',
}

def make_parser(kind, trace=0):
    if kind == 'rd':
        return nltk.RecursiveDescentParser(grammar, trace=trace)
    if kind == 'sr':
        return nltk.ShiftReduceParser(grammar, trace=trace)
    return nltk.ChartParser(grammar, trace=trace)

def parse_all(sentence, kind=None):
    parser = make_parser(kind or PARSER)
    return list(parser.parse(sentence.split()))

def get_trees(sentence):
    """Trees from the chosen parser; if it finds none, retry with the chart parser."""
    trees = parse_all(sentence, PARSER)
    if trees:
        return trees, PARSER
    return parse_all(sentence, 'chart'), 'chart (fallback)'

print(f'Parser chosen: {PARSER}')
for sid, s in sentences.items():
    trees, kind = get_trees(s)
    print(f'{sid}: {len(trees)} parse(s) using {kind}')

print('\nFor comparison, the number of parses found by each parser:')
print(pd.DataFrame({
    'sentence': list(sentences.values()),
    'num_parses': [len(parse_all(s, 'chart')) for s in sentences.values()]
}))


Parser chosen: chart
S1: 1 parse(s) using chart
S2: 1 parse(s) using chart
S3: 2 parse(s) using chart
S4: 1 parse(s) using chart

For comparison, the number of parses found by each parser:
                                          sentence  num_parses
0                 I saw the man with the telescope           1
1                             the dog chased a man           1
2  John saw the man in the park with the telescope           2
3                          Mary walked in the park           1


Step-by-step working of the parser on S4 (`trace`):


In [16]:
for sid, sentence in sentences.items():
    if sid == 'S4':
        print(f'\nParsing {sid}: {sentence}')
        trees = parse_all(sentence, PARSER)
        for tree in trees:
            print(tree)



Parsing S4: Mary walked in the park
(S
  (NP (PropN Mary))
  (VP (V walked) (PP (P in) (NP (Det the) (N park)))))


### (c) Display every parse tree: bracketed, text tree, and graphical


In [17]:
def show_tree(tree):
    print('Bracketed tree:')
    print(tree)
    print('\nIndented parse tree:')
    tree.pretty_print()

for sid, sentence in sentences.items():
    trees = parse_all(sentence, PARSER)
    if trees:
        print(f'\n=== {sid} ===')
        show_tree(trees[0])



=== S1 ===
Bracketed tree:
(S
  (NP (N I))
  (VP
    (V saw)
    (NP
      (NP (Det the) (N man))
      (PP (P with) (NP (Det the) (N telescope))))))

Indented parse tree:
     S                                    
  ___|___________                          
 |               VP                       
 |    ___________|___                      
 |   |               NP                   
 |   |        _______|____                 
 |   |       |            PP              
 |   |       |        ____|___             
 NP  |       NP      |        NP          
 |   |    ___|___    |     ___|______      
 N   V  Det      N   P   Det         N    
 |   |   |       |   |    |          |     
 I  saw the     man with the     telescope


=== S2 ===
Bracketed tree:
(S (NP (Det the) (N dog)) (VP (V chased) (NP (Det a) (N man))))

Indented parse tree:
              S               
      ________|_____           
     |              VP        
     |         _____|___       
     NP       |      

### (d) Ambiguity check


In [18]:
def is_ambiguous(sentence):
    """Return (number of distinct parse trees, True if more than one)."""
    trees = list(parse_all(sentence, PARSER))
    count = len(trees)
    return count, count > 1

for sid, sentence in sentences.items():
    count, ambiguous = is_ambiguous(sentence)
    print(f'{sid}: {count} parse(s), ambiguous={ambiguous}')


S1: 1 parse(s), ambiguous=False
S2: 1 parse(s), ambiguous=False
S3: 2 parse(s), ambiguous=True
S4: 1 parse(s), ambiguous=False


Inference
